<a href="https://colab.research.google.com/github/Andrew-MSU/web-gis-automations/blob/main/GPT_lecture_for_me.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Install dependencies
!pip install python-pptx openai gtts moviepy pdf2image

# Install poppler-utils and verify
!apt-get update
!apt-get install -y poppler-utils
!which pdfinfo  # Verify pdfinfo is installed
!pdfinfo -v  # Check pdfinfo version to confirm installation

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,338 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,666 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,235 kB]
Get:13 http://archive.ubuntu.com

In [9]:
import os
from pptx import Presentation
from openai import OpenAI
from gtts import gTTS
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips
from google.colab import files
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

# Define directories in Colab's temporary filesystem
BASE_DIR = "/content"
INPUT_DIR = os.path.join(BASE_DIR, "input")
IMAGES_DIR = os.path.join(BASE_DIR, "images")
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")
AUDIOS_DIR = os.path.join(BASE_DIR, "audios")
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")

# Create directories
for directory in [INPUT_DIR, IMAGES_DIR, SCRIPTS_DIR, AUDIOS_DIR, VIDEOS_DIR]:
    if not os.path.exists(directory):
        os.makedirs(directory)

print("Directories created:", INPUT_DIR, IMAGES_DIR, SCRIPTS_DIR, AUDIOS_DIR, VIDEOS_DIR)

Directories created: /content/input /content/images /content/scripts /content/audios /content/videos


### Instructions for Users
Open your PowerPoint file in Microsoft PowerPoint.
Export it as a PDF (File > Export > PDF). Ensure you export "All Slides" and use the standard layout.
Upload both the .pptx file and the .pdf file to Colab when prompted.

In [10]:
# Upload the PowerPoint file
print("Upload your PowerPoint file (e.g., sample.pptx):")
uploaded_ppt = files.upload()
ppt_filename = list(uploaded_ppt.keys())[0]
ppt_path = os.path.join(INPUT_DIR, ppt_filename)
with open(ppt_path, "wb") as f:
    f.write(uploaded_ppt[ppt_filename])

# Upload the PDF file
print("Upload the PDF version of your PowerPoint (e.g., sample.pdf):")
uploaded_pdf = files.upload()
pdf_filename = list(uploaded_pdf.keys())[0]
pdf_path = os.path.join(INPUT_DIR, pdf_filename)
with open(pdf_path, "wb") as f:
    f.write(uploaded_pdf[pdf_filename])

# Convert PDF pages to images
print("Converting PDF pages to images...")
images = convert_from_path(pdf_path)
for i, image in enumerate(images):
    image_path = os.path.join(IMAGES_DIR, f"slide_{i + 1}.png")
    image.save(image_path, "PNG")
    print(f"Saved slide image: {image_path}")

print(f"PowerPoint uploaded to: {ppt_path}")
print(f"PDF uploaded to: {pdf_path}")
print("Images generated in:", IMAGES_DIR)

Upload your PowerPoint file (e.g., sample.pptx):


Saving 1_Foundations.pptx to 1_Foundations (1).pptx
Upload the PDF version of your PowerPoint (e.g., sample.pdf):


Saving 1_Foundations.pdf to 1_Foundations (1).pdf
Converting PDF pages to images...
Saved slide image: /content/images/slide_1.png
Saved slide image: /content/images/slide_2.png
Saved slide image: /content/images/slide_3.png
Saved slide image: /content/images/slide_4.png
Saved slide image: /content/images/slide_5.png
Saved slide image: /content/images/slide_6.png
Saved slide image: /content/images/slide_7.png
Saved slide image: /content/images/slide_8.png
Saved slide image: /content/images/slide_9.png
Saved slide image: /content/images/slide_10.png
Saved slide image: /content/images/slide_11.png
Saved slide image: /content/images/slide_12.png
PowerPoint uploaded to: /content/input/1_Foundations (1).pptx
PDF uploaded to: /content/input/1_Foundations (1).pdf
Images generated in: /content/images


In [11]:
def parse_ppt(ppt_path):
    """Parse a PowerPoint file and extract text and images."""
    prs = Presentation(ppt_path)
    slide_data = []

    num_images = len([f for f in os.listdir(IMAGES_DIR) if f.startswith("slide_")])
    if num_images != len(prs.slides):
        print(f"Warning: Number of slides ({len(prs.slides)}) does not match number of PDF pages ({num_images})")

    for slide_num, slide in enumerate(prs.slides):
        slide_text = ""
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                slide_text += shape.text + "\n"
        notes_text = ""
        if slide.notes_slide.notes_text_frame:
            notes_text = slide.notes_slide.notes_text_frame.text
        slide_image_path = os.path.join(IMAGES_DIR, f"slide_{slide_num + 1}.png")
        if not os.path.exists(slide_image_path):
            print(f"Warning: Image for slide {slide_num + 1} not found at {slide_image_path}")
            continue
        slide_data.append({
            "slide_num": slide_num + 1,
            "text": slide_text,
            "notes": notes_text,
            "image_path": slide_image_path
        })

    return slide_data

slides = parse_ppt(ppt_path)
for slide in slides:
    print(f"Slide {slide['slide_num']}:")
    print(f"Text: {slide['text']}")
    print(f"Notes: {slide['notes']}")
    print(f"Image Path: {slide['image_path']}\n")

Slide 1:
Text: GEO 315: StructureIntroductions, Outlineand Foundations
Dr. Andrew Laskowski (Drew)
Associate Professor, Earth Sciences

Notes: 
Image Path: /content/images/slide_1.png

Slide 2:
Text: Questions
2

Notes: 
Image Path: /content/images/slide_2.png

Slide 3:
Text: Warm-Up Activity
In your notes, sketch the following:

A block diagram of a normal fault, reverse fault, and strike-slip fault
A cross section of an anticline and a syncline

Define the following:

Stress
Strain
Kinematics
Dynamics
Introduction&Foundations
3

Notes: 
Image Path: /content/images/slide_3.png

Slide 4:
Text: Stratigraphy
Introduction&Foundations
4
© Ben van der Pluijm
Stratigraphic Principles
Principle of original horizontality.
Principle of superposition. 
Discontinuous sedimentation
Erosional Surfaces
Slumps and Slides
Unconformities
Stratigraphic succession of the Colorado Plateau, US 

Notes: 
Image Path: /content/images/slide_4.png

Slide 5:
Text: Stratigraphy: Unconformities
Introduction&Fou

In [16]:
from google.colab import userdata

# Retrieve the API key from Colab Secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def generate_initial_script(slide_text, notes_text, slide_num):
    """Generate an initial script for a slide using OpenAI."""
    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = (
        "You are a lecturer preparing a script for a slide in a presentation. "
        "The slide content is:\n\n"
        f"Slide Text: {slide_text}\n"
        f"Speaker Notes: {notes_text}\n\n"
        "Please generate a clear and concise script that explains the slide in a way suitable for a lecture, "
        "covering key points in a first-person tone. Keep it under 100 words."
    )
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Using gpt-3.5-turbo as fallback
            messages=[
                {"role": "system", "content": "You are a helpful lecturer."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150
        )
        script = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating initial script for Slide {slide_num}: {str(e)}")
        script = "Failed to generate initial script due to an error."

    # Save initial script temporarily
    initial_script_path = os.path.join(SCRIPTS_DIR, f"initial_script_slide_{slide_num}.txt")
    with open(initial_script_path, "w", encoding="utf-8") as f:
        f.write(script)

    return script, initial_script_path

# Generate initial scripts for each slide and collect them
initial_scripts = []
for slide in slides:
    print(f"Generating initial script for Slide {slide['slide_num']}...")
    script, script_path = generate_initial_script(slide["text"], slide["notes"], slide["slide_num"])
    print(f"Initial script saved to: {script_path}")
    print(f"Initial script: {script}\n")
    initial_scripts.append({
        "slide_num": slide["slide_num"],
        "text": slide["text"],
        "notes": slide["notes"],
        "initial_script": script,
        "initial_script_path": script_path
    })

Generating initial script for Slide 1...
Initial script saved to: /content/scripts/initial_script_slide_1.txt
Initial script: Hello, everyone. I'm Dr. Andrew Laskowski, but most people call me Drew. Today, we are delving into GEO 315: Structure – where we’ll cover introductions, outline, and foundational concepts. As an Associate Professor in Earth Sciences, I'm excited to guide you through this journey. We’ll explore the fundamental aspects that form the backbone of understanding geological structures. Let’s dive in together and lay a solid groundwork for our exploration in this course. Let's start by establishing a strong foundation for our learning.

Generating initial script for Slide 2...
Initial script saved to: /content/scripts/initial_script_slide_2.txt
Initial script: "Welcome to the next section of our presentation, focusing on questions. When discussing any topic, it's essential to encourage questions from the audience. Questions promote engagement, deepen understanding, and

In [17]:
def refine_scripts_with_coherence(initial_scripts):
    """Refine scripts to ensure coherence across the presentation using OpenAI."""
    client = OpenAI(api_key=OPENAI_API_KEY)

    # Combine all initial scripts into a single context string
    full_context = "Below is the draft of a presentation with scripts for each slide:\n\n"
    for item in initial_scripts:
        full_context += (
            f"Slide {item['slide_num']}:\n"
            f"Slide Text: {item['text']}\n"
            f"Speaker Notes: {item['notes']}\n"
            f"Initial Script: {item['initial_script']}\n\n"
        )

    refined_scripts = []
    for idx, item in enumerate(initial_scripts):
        slide_num = item['slide_num']
        print(f"Refining script for Slide {slide_num}...")

        # Create a prompt that includes the full context and instructions to refine
        prompt = (
            f"I am a lecturer preparing a script for a coherent presentation. "
            "Below is the full draft of my presentation with initial scripts for each slide:\n\n"
            f"{full_context}\n\n"
            f"Now, refine the script for Slide {slide_num} to ensure it flows naturally within the presentation. "
            "Consider what has been discussed in previous slides and preview what is upcoming in future slides. "
            "Add transitions if necessary to connect with prior and upcoming content. "
            "Keep the tone professional, first-person, and concise (under 100 words). "
            "Focus on the content of Slide {slide_num}, but make it part of a cohesive lecture."
        )

        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",  # Using gpt-3.5-turbo as fallback
                messages=[
                    {"role": "system", "content": "You are a helpful lecturer ensuring a cohesive presentation."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=150
            )
            refined_script = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error refining script for Slide {slide_num}: {str(e)}")
            refined_script = item['initial_script']  # Fallback to initial script if refinement fails

        # Save refined script
        script_path = os.path.join(SCRIPTS_DIR, f"script_slide_{slide_num}.txt")
        with open(script_path, "w", encoding="utf-8") as f:
            f.write(refined_script)

        print(f"Refined script saved to: {script_path}")
        print(f"Refined script: {refined_script}\n")

        refined_scripts.append({
            "slide_num": slide_num,
            "text": item["text"],
            "notes": item["notes"],
            "refined_script": refined_script,
            "script_path": script_path
        })

    return refined_scripts

# Refine scripts for coherence
refined_scripts = refine_scripts_with_coherence(initial_scripts)

Refining script for Slide 1...
Refined script saved to: /content/scripts/script_slide_1.txt
Refined script: Hello, everyone. I'm Dr. Andrew Laskowski, but feel free to call me Drew. We've already explored essential warm-up activities and foundational terms in GEO 315. Now, as we transition to discussing questions in Slide 2, remember that curiosity and engagement drive our learning experience. In upcoming slides, we'll journey through stratigraphy, uncovering stratigraphic principles and the fascinating world of unconformities. Stay engaged as we navigate deformation regimes and delve into concepts like strike and dip. Let's continue building on the groundwork we've established and unravel the intricate layers of our geological exploration.

Refining script for Slide 2...
Refined script saved to: /content/scripts/script_slide_2.txt
Refined script: Slide 2:

Initial Script Refinement:
"As we dive into the essence of geological structures, it's crucial to foster an environment where ques

In [18]:
def generate_audio(script_path, slide_num):
    """Generate audio from a script using gTTS."""
    with open(script_path, "r", encoding="utf-8") as f:
        script = f.read()
    tts = gTTS(text=script, lang="en")
    audio_path = os.path.join(AUDIOS_DIR, f"audio_slide_{slide_num}.mp3")
    tts.save(audio_path)
    return audio_path

for slide in slides:
    script_path = os.path.join(SCRIPTS_DIR, f"script_slide_{slide['slide_num']}.txt")
    print(f"Generating audio for Slide {slide['slide_num']}...")
    audio_path = generate_audio(script_path, slide["slide_num"])
    print(f"Audio saved to: {audio_path}\n")

Generating audio for Slide 1...
Audio saved to: /content/audios/audio_slide_1.mp3

Generating audio for Slide 2...
Audio saved to: /content/audios/audio_slide_2.mp3

Generating audio for Slide 3...
Audio saved to: /content/audios/audio_slide_3.mp3

Generating audio for Slide 4...
Audio saved to: /content/audios/audio_slide_4.mp3

Generating audio for Slide 5...
Audio saved to: /content/audios/audio_slide_5.mp3

Generating audio for Slide 6...
Audio saved to: /content/audios/audio_slide_6.mp3

Generating audio for Slide 7...
Audio saved to: /content/audios/audio_slide_7.mp3

Generating audio for Slide 8...
Audio saved to: /content/audios/audio_slide_8.mp3

Generating audio for Slide 9...
Audio saved to: /content/audios/audio_slide_9.mp3

Generating audio for Slide 10...
Audio saved to: /content/audios/audio_slide_10.mp3

Generating audio for Slide 11...
Audio saved to: /content/audios/audio_slide_11.mp3

Generating audio for Slide 12...
Audio saved to: /content/audios/audio_slide_12.mp3

In [ ]:
def assemble_video(slides):
    clips = []
    for slide in slides:
        slide_num = slide["slide_num"]
        image_path = slide["image_path"]
        audio_path = os.path.join(AUDIOS_DIR, f"audio_slide_{slide_num}.mp3")
        if not os.path.exists(image_path):
            print(f"Error: Image for slide {slide_num} not found. Skipping...")
            continue
        audio = AudioFileClip(audio_path)
        duration = audio.duration
        image_clip = ImageClip(image_path, duration=duration)
        image_clip = image_clip.set_audio(audio)
        clips.append(image_clip)
    if not clips:
        raise ValueError("No valid clips to concatenate.")
    final_clip = concatenate_videoclips(clips, method="compose")
    output_path = os.path.join(VIDEOS_DIR, "lecture_video.mp4")
    final_clip.write_videofile(output_path, fps=24)
    return output_path

video_path = assemble_video(slides)

Moviepy - Building video /content/videos/lecture_video.mp4.
MoviePy - Writing audio in lecture_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video /content/videos/lecture_video.mp4



t:   1%|▏         | 210/14367 [00:27<40:59,  5.75it/s, now=None]

In [ ]:
files.download(video_path)